# 🎸 FretFlow Audio Engine — Real-Time Inference PoC
**Low-Latency Chord Recognition | Signal Processing & Edge AI**

This notebook demonstrates the deterministic audio pipeline designed for sub-20ms chord recognition on mobile devices. It bypasses server-side latency by processing audio buffers locally through a quantized feature extraction and ML classification loop.

| Component | Tech Stack | Role |
|-----------|------------|------|
| **Input Buffer** | Numpy / Circular Buffer | Real-time stream management (1024-sample window) |
| **Feature Extractor** | Librosa / CQT | Mathematical spectral compression (Acoustic DNA) |
| **ML Inference** | Core ML / TFLite (Simulated) | High-speed classification on the Neural Engine |
| **DSP Effects** | Pedalboard | Signal conditioning (Normalization / Gain) |

> **Status:** Pipeline optimized for sub-20ms target latency.

## ⚙️ CELL 1 — Setup & Buffer Simulation
We treat audio as a live stream, not a file. This cell initializes the environment and simulates a high-frequency circular buffer (e.g., 1024 samples at 44.1kHz).

In [1]:
import numpy as np
import librosa
import time
import warnings
from pedalboard import Pedalboard, Gain
warnings.filterwarnings("ignore")

SAMPLE_RATE = 44100
BUFFER_SIZE = 1024

def simulate_mic_buffer(size=BUFFER_SIZE):
    """Simulates a raw microphone buffer capture."""
    return np.random.uniform(-0.1, 0.1, size).astype(np.float32)

# Initialize a lightweight DSP board for signal conditioning
board = Pedalboard([Gain(gain_db=6.0)])

input_buffer = simulate_mic_buffer()
conditioned_output = board(input_buffer, sample_rate=SAMPLE_RATE)

print(f"✅ Environment Ready. Input shape: {input_buffer.shape}")
print("🔊 Signal conditioned (Gain +6dB Applied).")

✅ Environment Ready. Input shape: (1024,)
🔊 Signal conditioned (Gain +6dB Applied).


## 🧠 CELL 2 — Spectral Feature Extraction (The Brain)
We convert the raw time-domain buffer into the frequency domain using a Short-Time Fourier Transform (STFT) mapped to a Chroma vector. This represents the 'Energy' of each note (A through G#) in the current buffer.

In [2]:
def extract_fretflow_dna(buffer, sr=SAMPLE_RATE):
    # Extract Chroma features to identify note energy
    # Optimization: Use a single frame for sub-20ms simulation
    chroma = librosa.feature.chroma_stft(y=buffer, sr=sr, n_fft=BUFFER_SIZE, hop_length=BUFFER_SIZE+1)
    return np.mean(chroma, axis=1)

dna_vector = extract_fretflow_dna(conditioned_output)
print(f"📊 Extracted Acoustic DNA (Chroma Vector):\n{dna_vector}")
print("\n💡 This 12-dimension vector is what feeds the classifier, not the raw audio.")

📊 Extracted Acoustic DNA (Chroma Vector):
[1.         0.6793056  0.6602736  0.86371857 0.69576734 0.5220398
 0.5540362  0.703292   0.75179654 0.7228358  0.6968413  0.7370818 ]

💡 This 12-dimension vector is what feeds the classifier, not the raw audio.


## ⏱️ CELL 3 — Latency Benchmark
Performance is the primary constraint. This cell benchmarks the full loop (Buffer -> DSP -> Feature Extraction) to ensure we stay within the sub-20ms interactive window.

In [3]:
iterations = 200
latencies = []

for _ in range(iterations):
    start = time.perf_counter()
    
    # 1. Capture
    raw = simulate_mic_buffer()
    # 2. Condition
    conditioned = board(raw, sample_rate=SAMPLE_RATE)
    # 3. Extract
    _ = extract_fretflow_dna(conditioned)
    
    end = time.perf_counter()
    latencies.append((end - start) * 1000)

avg_latency = np.mean(latencies)
p95_latency = np.percentile(latencies, 95)

print(f"🚀 Benchmark Results ({iterations} iterations):")
print(f"   Average Loop Latency: {avg_latency:.2f} ms")
print(f"   P95 Latency: {p95_latency:.2f} ms")

if avg_latency < 20:
    print("\n✅ SUCCESS: Pipeline meets sub-20ms real-time requirements.")
else:
    print("\n⚠️ WARNING: Latency exceeds 20ms. Optimization required.")

🚀 Benchmark Results (200 iterations):
   Average Loop Latency: 1.23 ms
   P95 Latency: 1.66 ms

✅ SUCCESS: Pipeline meets sub-20ms real-time requirements.
